In [1]:
import os
import zipfile
import platform
from datetime import datetime

def formatar_caminho_longo(caminho):
    """
    Adiciona o prefixo '\\?\' no Windows para contornar o limite de 260 caracteres (MAX_PATH).
    """
    if platform.system() == 'Windows':
        caminho_absoluto = os.path.abspath(caminho)
        if not caminho_absoluto.startswith('\\\\?\\'):
            return '\\\\?\\' + caminho_absoluto
    return caminho

def extrair_e_deletar_zips(diretorio_base):
    # Aplica a correção de caminho longo na raiz para que os subdiretórios herdem
    diretorio_base_formatado = formatar_caminho_longo(diretorio_base)
    
    print(f"🔍 Buscando arquivos ZIP em: {diretorio_base}")
    
    # Contadores e listas para o relatório
    arquivos_modificados = []
    total_sucesso = 0
    total_erros = 0
    
    # =========================================================================
    # SOLUÇÃO PARA ZIPS ANINHADOS: 
    # O loop 'while' continua rodando enquanto houver arquivos ZIP sendo encontrados
    # =========================================================================
    zips_pendentes = True
    
    while zips_pendentes:
        zips_pendentes = False # Assume que acabou, a menos que encontre um novo zip
        
        # walk percorre todas as subpastas
        for root, dirs, files in os.walk(diretorio_base_formatado):
            for file in files:
                if file.lower().endswith('.zip'):
                    # Encontrou um zip! Extrai agora, mas avisa o while para rodar de novo
                    zips_pendentes = True 
                    
                    caminho_zip = os.path.join(root, file)
                    
                    # Define a pasta de destino com o mesmo nome do arquivo zip
                    nome_pasta = os.path.splitext(file)[0]
                    pasta_destino = os.path.join(root, nome_pasta)

                    print(f"\n📦 Processando: {file}")
                    
                    try:
                        # Cria a pasta de destino se não existir
                        if not os.path.exists(pasta_destino):
                            os.makedirs(pasta_destino)
                        
                        # Abre e extrai
                        with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
                            # Extrai todos os arquivos lidando com os caminhos longos
                            zip_ref.extractall(pasta_destino)
                        
                        print(f"✅ Extraído para: {pasta_destino}")
                        
                        # Deleta o arquivo zip original
                        os.remove(caminho_zip)
                        print(f"🗑️ Zip original removido.")
                        
                        # -----------------------------------------------------
                        # REGISTRO PARA O RELATÓRIO
                        # (Removemos o prefixo '\\?\' se existir)
                        caminho_exibicao = caminho_zip
                        if caminho_exibicao.startswith('\\\\?\\'):
                            caminho_exibicao = caminho_exibicao[4:]
                            
                        pasta_exibicao = pasta_destino
                        if pasta_exibicao.startswith('\\\\?\\'):
                            pasta_exibicao = pasta_exibicao[4:]
                        
                        arquivos_modificados.append({
                            'nome': file,
                            'caminho_original': caminho_exibicao,
                            'pasta_extraida': pasta_exibicao
                        })
                        total_sucesso += 1
                        # -----------------------------------------------------
                        
                    except PermissionError as pe:
                        print(f"🔒 Erro de Permissão/Segurança ao processar {file}: {pe}")
                        print("O Windows ou Antivírus pode estar bloqueando o arquivo, ou ele está em uso.")
                        total_erros += 1
                        
                    except zipfile.BadZipFile:
                        print(f"⚠️ Arquivo corrompido: {file}")
                        print("O arquivo ZIP não é válido ou está incompleto e foi mantido.")
                        total_erros += 1
                        
                    except Exception as e:
                        print(f"❌ Erro crítico ao processar {file}: {e}")
                        print("O arquivo ZIP foi mantido para segurança.")
                        total_erros += 1

    print(f"\n🏁 Processo de varredura concluído.")
    print(f"📊 Resumo: {total_sucesso} substituídos com sucesso | {total_erros} erros detectados.")
    
    # =========================================================================
    # GERAÇÃO DO ARQUIVO TXT (RELATÓRIO)
    # =========================================================================
    if arquivos_modificados:
        data_hora_atual = datetime.now().strftime('%Y%m%d_%H%M%S')
        nome_relatorio = f"relatorio_modificacoes_{data_hora_atual}.txt"
        
        caminho_relatorio = os.path.join(os.path.abspath(diretorio_base), nome_relatorio)
        caminho_relatorio_formatado = formatar_caminho_longo(caminho_relatorio)
        
        try:
            with open(caminho_relatorio_formatado, 'w', encoding='utf-8') as f:
                f.write(f"=== RELATÓRIO DE SUBSTITUIÇÃO DE ARQUIVOS ZIP ===\n")
                f.write(f"Data e Hora do Início: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n")
                f.write(f"Diretório Raiz: {os.path.abspath(diretorio_base)}\n")
                f.write(f"Total de arquivos ZIP processados e excluídos com sucesso: {total_sucesso}\n")
                f.write(f"Total de falhas (mantidos intocados): {total_erros}\n")
                f.write(f"=================================================\n\n")
                
                f.write(f"--- LISTA DE ARQUIVOS MODIFICADOS ---\n\n")
                for idx, item in enumerate(arquivos_modificados, 1):
                    f.write(f"{idx}. Arquivo: {item['nome']}\n")
                    f.write(f"   De (Caminho Zip): {item['caminho_original']}\n")
                    f.write(f"   Para (Nova Pasta): {item['pasta_extraida']}\n")
                    f.write(f"   Status: ZIP original deletado; pasta criada com sucesso.\n")
                    f.write(f"-------------------------------------------------\n")
            
            print(f"📝 Um relatório detalhado foi gerado com sucesso em:\n -> {caminho_relatorio}")
        except Exception as e:
            print(f"❌ Erro ao tentar gerar o arquivo de relatório TXT: {e}")
    else:
        if total_erros > 0:
             print("ℹ️ Houve tentativas, mas nenhum arquivo foi modificado devido a erros. Nenhum relatório gerado.")
        else:
             print("ℹ️ Nenhum arquivo .zip foi encontrado, portanto nenhum relatório foi gerado.")

if __name__ == "__main__":
    # Pega o diretório onde o script está salvo (neste caso, a raiz do projeto se o .ipynb estiver lá)
    diretorio_atual = os.getcwd()
    extrair_e_deletar_zips(diretorio_atual)

🔍 Buscando arquivos ZIP em: c:\Users\e-giuseppe.parrini\Documents\estudos-python

📦 Processando: 1. Lendo Arquivos txt.zip
✅ Extraído para: \\?\c:\Users\e-giuseppe.parrini\Documents\estudos-python\3-intermediario\24. Integração Python com Arquivos txt e PDF\1. Lendo Arquivos txt
🗑️ Zip original removido.

📦 Processando: 10. Integração Python e PDF - Como funciona.zip
✅ Extraído para: \\?\c:\Users\e-giuseppe.parrini\Documents\estudos-python\3-intermediario\24. Integração Python com Arquivos txt e PDF\10. Integração Python e PDF - Como funciona
🗑️ Zip original removido.

📦 Processando: 11. Separar páginas e Criar PDF.zip
✅ Extraído para: \\?\c:\Users\e-giuseppe.parrini\Documents\estudos-python\3-intermediario\24. Integração Python com Arquivos txt e PDF\11. Separar páginas e Criar PDF
🗑️ Zip original removido.

📦 Processando: 8. Mentoria - Leitura de XML e Notas Fiscais com Python.zip
✅ Extraído para: \\?\c:\Users\e-giuseppe.parrini\Documents\estudos-python\3-intermediario\24. Integração